Import Libraries

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

Libraries imported successfully!


##### Load Dataset

In [4]:
# Load the dataset
df = pd.read_csv('../data/RTA Dataset.csv')

print("Dataset Loaded Successfully!")


Dataset Loaded Successfully!


In [5]:
print("Shape of dataset:", df.shape)
print("\nFirst 5 rows:")
display(df.head())

print("\nColumns in dataset:")
print(df.columns.tolist())

Shape of dataset: (12316, 25)

First 5 rows:


,Time,Day_of_week,Age_band_of_driver,Sex_of_driver,Educational_level,Vehicle_driver_relation,Driving_experience,Type_of_vehicle,Owner_of_vehicle,Service_year_of_vehicle,...,Road_surface_type,Road_surface_conditions,Light_conditions,Weather_conditions,Type_of_collision,Number_of_vehicles_involved,Vehicle_movement,Casualty_class,Sex_of_casualty,Cause_of_accident
0,17:02:00,Monday,18-30,Male,Above high school,Employee,1-2yr,Automobile,Owner,Above 10yr,...,Asphalt roads,Dry,Daylight,Normal,Collision with roadside-parked vehicles,2,Going straight,na,na,Moving Backward
1,17:02:00,Monday,31-50,Male,Junior high school,Employee,Above 10yr,Public (> 45 seats),Owner,5-10yrs,...,Asphalt roads,Dry,Daylight,Normal,Vehicle with vehicle collision,2,Going straight,na,na,Overtaking
2,17:02:00,Monday,18-30,Male,Junior high school,Employee,1-2yr,Lorry (41?100Q),Owner,NaN,...,Asphalt roads,Dry,Daylight,Normal,Collision with roadside objects,2,Going straight,Driver or rider,Male,Changing lane to the left
3,01:06:00,Sunday,18-30,Male,Junior high school,Employee,5-10yr,Public (> 45 seats),Governmental,NaN,...,Earth roads,Dry,Darkness - lights lit,Normal,Vehicle with vehicle collision,2,Going straight,Pedestrian,Female,Changing lane to the right
4,01:06:00,Sunday,18-30,Male,Junior high school,Employee,2-5yr,NaN,Owner,5-10yrs,...,Asphalt roads,Dry,Darkness - lights lit,Normal,Vehicle with vehicle collision,2,Going straight,na,na,Overtaking



Columns in dataset:
['Time', 'Day_of_week', 'Age_band_of_driver', 'Sex_of_driver', 'Educational_level', 'Vehicle_driver_relation', 'Driving_experience', 'Type_of_vehicle', 'Owner_of_vehicle', 'Service_year_of_vehicle', 'Defect_of_vehicle', 'Area_accident_occured', 'Lanes_or_Medians', 'Road_allignment', 'Types_of_Junction', 'Road_surface_type', 'Road_surface_conditions', 'Light_conditions', 'Weather_conditions', 'Type_of_collision', 'Number_of_vehicles_involved', 'Vehicle_movement', 'Casualty_class', 'Sex_of_casualty', 'Cause_of_accident']


In [6]:
columns_to_drop = [
    'Age_band_of_driver',
    'Sex_of_driver',
    'Educational_level',
    'Vehicle_driver_relation',
    'Driving_experience',
    'Owner_of_vehicle',
    'Service_year_of_vehicle',
    'Sex_of_casualty',
    'Cause_of_accident',
    'Type_of_vehicle',
	'Defect_of_vehicle',
	'Road_allignment',
	'Road_surface_type',
	'Vehicle_movement',
]

# Drop only if they exist (safe)
df = df.drop(columns=[col for col in columns_to_drop if col in df.columns])

Check data types

Check remaining columns to get all unique values

Overview

Check Missing Values

Visualize Missing Values

Remove Rows with Missing Target (Casualty_class + Type_of_collision + Number_of_vehicles_involved)

In [7]:
df.shape

(12316, 11)

In [8]:
# Count missing values per column
missing_values = df.isnull().sum()

# Show only columns with missing values
missing_values

Time                             0
Day_of_week                      0
Area_accident_occured          239
Lanes_or_Medians               385
Types_of_Junction              887
Road_surface_conditions          0
Light_conditions                 0
Weather_conditions               0
Type_of_collision              155
Number_of_vehicles_involved      0
Casualty_class                   0
dtype: int64

In [9]:
# Replace all remaining NaN values
df = df.fillna("Unknown")

In [10]:
# Check again for missing values
df.isnull().sum()

Time                           0
Day_of_week                    0
Area_accident_occured          0
Lanes_or_Medians               0
Types_of_Junction              0
Road_surface_conditions        0
Light_conditions               0
Weather_conditions             0
Type_of_collision              0
Number_of_vehicles_involved    0
Casualty_class                 0
dtype: int64

In [11]:
# Check cleaned dataset
df.head()

print("\nFinal Shape:", df.shape)
print("\nRemaining Columns:\n", df.columns.tolist())


Final Shape: (12316, 11)

Remaining Columns:
 ['Time', 'Day_of_week', 'Area_accident_occured', 'Lanes_or_Medians', 'Types_of_Junction', 'Road_surface_conditions', 'Light_conditions', 'Weather_conditions', 'Type_of_collision', 'Number_of_vehicles_involved', 'Casualty_class']


In [12]:
# Get combinations with counts
combinations = df.groupby(['Casualty_class', 
                           'Type_of_collision', 
                           'Number_of_vehicles_involved']).size().reset_index(name='Count')

# Sort by frequency (most common first)
combinations = combinations.sort_values(by='Count', ascending=False)

# Display all combinations
print(f"Total Unique Combinations: {len(combinations)}\n")
print(combinations.to_string(index=False))

Total Unique Combinations: 142

 Casualty_class                       Type_of_collision  Number_of_vehicles_involved  Count
Driver or rider          Vehicle with vehicle collision                            2   2321
             na          Vehicle with vehicle collision                            2   2141
     Pedestrian          Vehicle with vehicle collision                            2    816
      Passenger          Vehicle with vehicle collision                            2    624
Driver or rider          Vehicle with vehicle collision                            1    589
Driver or rider         Collision with roadside objects                            2    542
             na          Vehicle with vehicle collision                            1    505
Driver or rider          Vehicle with vehicle collision                            3    456
             na          Vehicle with vehicle collision                            3    441
             na         Collision with roadside 

In [13]:
# Create Risk Level 
def get_risk_level_manual(row):
    casualty = str(row['Casualty_class']).strip().lower() if pd.notna(row['Casualty_class']) else 'na'
    collision = str(row['Type_of_collision']).strip().lower() if pd.notna(row['Type_of_collision']) else ''
    vehicles = row['Number_of_vehicles_involved']
    
    try:
        vehicles = int(vehicles)
    except:
        vehicles = 2  # default
    
    # High Risk (H) - Based on your labeling
    if 'pedestrian' in casualty:
        return 'High'
    elif 'rollover' in collision or 'train' in collision:
        return 'High'
    elif vehicles >= 4:
        return 'High'
    elif casualty == 'na' and vehicles >= 3 and 'vehicle with vehicle' in collision:
        return 'High'
    elif 'collision with roadside objects' in collision and vehicles >= 2 and casualty in ['driver or rider', 'na']:
        return 'High'
    
    # Medium Risk (M)
    elif casualty == 'passenger':
        return 'Medium'
    elif vehicles == 3:
        return 'Medium'
    elif 'vehicle with vehicle' in collision and vehicles == 2 and casualty == 'na':
        return 'Medium'
    elif 'roadside' in collision and vehicles == 2:
        return 'Medium'
    elif casualty == 'driver or rider' and vehicles == 1:
        return 'Medium'
    
    # Low Risk (L) - Default for safer cases
    else:
        return 'Low'

# Apply the function
df['Risk_Level'] = df.apply(get_risk_level_manual, axis=1)

# Check final distribution
print("Final Risk Level Distribution:")
print(df['Risk_Level'].value_counts())
print("\nPercentage:")
print(round(df['Risk_Level'].value_counts(normalize=True) * 100, 2))

Final Risk Level Distribution:
Risk_Level
Medium    4724
High      3924
Low       3668
Name: count, dtype: int64

Percentage:
Risk_Level
Medium    38.36
High      31.86
Low       29.78
Name: proportion, dtype: float64


In [14]:
# Count missing values per column
missing_values = df.isnull().sum()

# Show only columns with missing values
missing_values

Time                           0
Day_of_week                    0
Area_accident_occured          0
Lanes_or_Medians               0
Types_of_Junction              0
Road_surface_conditions        0
Light_conditions               0
Weather_conditions             0
Type_of_collision              0
Number_of_vehicles_involved    0
Casualty_class                 0
Risk_Level                     0
dtype: int64

##### remove unwanted features

##### convert time to morning, afternoon, evening and night

In [15]:
df.head()

,Time,Day_of_week,Area_accident_occured,Lanes_or_Medians,Types_of_Junction,Road_surface_conditions,Light_conditions,Weather_conditions,Type_of_collision,Number_of_vehicles_involved,Casualty_class,Risk_Level
0,17:02:00,Monday,Residential areas,Unknown,No junction,Dry,Daylight,Normal,Collision with roadside-parked vehicles,2,na,Medium
1,17:02:00,Monday,Office areas,Undivided Two way,No junction,Dry,Daylight,Normal,Vehicle with vehicle collision,2,na,Medium
2,17:02:00,Monday,Recreational areas,other,No junction,Dry,Daylight,Normal,Collision with roadside objects,2,Driver or rider,High
3,01:06:00,Sunday,Office areas,other,Y Shape,Dry,Darkness - lights lit,Normal,Vehicle with vehicle collision,2,Pedestrian,High
4,01:06:00,Sunday,Industrial areas,other,Y Shape,Dry,Darkness - lights lit,Normal,Vehicle with vehicle collision,2,na,Medium
